In [ ]:
# ==========================================
# TRANSFORMACION FLUIDOS FRUF19
# Google Colab / Python
# ==========================================

import pandas as pd
import numpy as np
import re

# ------------------------------------------
# Archivo origen
# ------------------------------------------

archivo_entrada = "/content/Fluidos_FRUf19_orig.xlsx"
archivo_salida = "Fluidos_FRUf19_Master_2.xlsx"

# ==========================================
# HOJA PROPS&CONC
# ==========================================

pc = pd.read_excel(
    archivo_entrada,
    sheet_name="Props&Conc",
    header=None,
    engine="openpyxl"
)

# Fechas
fechas = []

for c in range(2, pc.shape[1]):
    fecha = pd.to_datetime(pc.iat[0, c], errors="coerce")

    if pd.notna(fecha):
        fechas.append(fecha.strftime("%Y-%m-%d"))
    else:
        fechas.append(None)

# Reportes
reportes = [
    pc.iat[1, c]
    for c in range(2, pc.shape[1])
]

# DataFrames base
operaciones = {
    "Fecha": fechas,
    "Reporte": reportes
}

propiedades = {
    "Fecha": fechas,
    "Reporte": reportes
}

concentraciones = {
    "Fecha": fechas,
    "Reporte": reportes
}

# Recorrer variables
for r in range(2, pc.shape[0]):

    variable = str(pc.iat[r, 0]).strip()
    tipo = str(pc.iat[r, 1]).strip().lower()

    valores = [
        pc.iat[r, c]
        for c in range(2, pc.shape[1])
    ]

    if tipo == "operaciones":
        operaciones[variable] = valores

    elif tipo == "propiedades":
        propiedades[variable] = valores

    elif tipo == "concentraciones":
        concentraciones[variable] = valores

# Convertir a DataFrame

df_operaciones = pd.DataFrame(operaciones)
df_propiedades = pd.DataFrame(propiedades)
df_concentraciones = pd.DataFrame(concentraciones)

# Eliminar fechas duplicadas
df_operaciones = df_operaciones.drop_duplicates(
    subset="Fecha",
    keep="last"
)

df_propiedades = df_propiedades.drop_duplicates(
    subset="Fecha",
    keep="last"
)

df_concentraciones = df_concentraciones.drop_duplicates(
    subset="Fecha",
    keep="last"
)

# ==========================================
# HOJA VOLUMENES
# ==========================================

vol = pd.read_excel(
    archivo_entrada,
    sheet_name="Volumenes",
    header=None,
    engine="openpyxl"
)

registros = []

for c in range(2, vol.shape[1]):

    fecha_raw = vol.iat[1, c]
    fecha = pd.to_datetime(fecha_raw, errors="coerce")

    if pd.isna(fecha):
        # Attempt to parse 'DD MonthName' format if the first attempt failed
        month_map = {
            'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
            'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
        }

        match = re.match(r'(\d+)\s+([A-Za-z]+)', str(fecha_raw))
        if match:
            day = int(match.group(1))
            month_name = match.group(2)
            month_name_cleaned = month_name.capitalize() # Ensure first letter is capitalized

            if month_name_cleaned in month_map:
                month = month_map[month_name_cleaned]
                year = 2026 # Assuming year 2026 based on previous valid dates
                date_str = f"{year}-{month:02d}-{day:02d}"
                fecha = pd.to_datetime(date_str, errors="coerce")

    if pd.isna(fecha): # If still NaT after all attempts
        # No need to print 'Saltando columna...' for debugging dates, as issue is resolved
        continue

    reporte_raw = str(vol.iat[2, c])

    # Extract only the numbers from the report string
    numeric_reporte_match = re.search(r'\d+', reporte_raw)
    if numeric_reporte_match:
        reporte_final = numeric_reporte_match.group(0)
    else:
        # If no numeric part is found, result in an empty string
        reporte_final = ''

    # Debugging: print what was processed for the report number
    if reporte_raw != reporte_final:
        print(f"Reporte original en Volumenes: '{reporte_raw}', Reporte extraído: '{reporte_final}'")

    fila = {
        "Fecha": fecha.strftime("%Y-%m-%d"),
        "Reporte": reporte_final
    }

    for r in range(3, vol.shape[0]):

        nombre_variable = vol.iat[r, 0]

        if pd.notna(nombre_variable):
            fila[str(nombre_variable).strip()] = vol.iat[r, c]

    registros.append(fila)

df_volumenes = pd.DataFrame(registros)

df_volumenes = df_volumenes.drop_duplicates(
    subset="Fecha",
    keep="last"
)

# ==========================================
# EXPORTAR EXCEL
# ==========================================

with pd.ExcelWriter(
    archivo_salida,
    engine="openpyxl"
) as writer:

    df_volumenes.to_excel(
        writer,
        sheet_name="Volumenes",
        index=False
    )

    df_operaciones.to_excel(
        writer,
        sheet_name="Operaciones",
        index=False
    )

    df_propiedades.to_excel(
        writer,
        sheet_name="Propiedades",
        index=False
    )

    df_concentraciones.to_excel(
        writer,
        sheet_name="Concentraciones",
        index=False
    )

print("Archivo generado:")
print(archivo_salida)


Reporte original en Volumenes: 'REPORTE #1', Reporte extraído: '1'
Reporte original en Volumenes: 'REPORTE #2', Reporte extraído: '2'
Reporte original en Volumenes: 'REPORTE #3', Reporte extraído: '3'
Reporte original en Volumenes: 'REPORTE #4', Reporte extraído: '4'
Reporte original en Volumenes: 'REPORTE #5', Reporte extraído: '5'
Reporte original en Volumenes: 'REPORTE #6', Reporte extraído: '6'
Reporte original en Volumenes: 'REPORTE #7', Reporte extraído: '7'
Reporte original en Volumenes: 'REPORTE #8', Reporte extraído: '8'
Reporte original en Volumenes: 'REPORTE #9', Reporte extraído: '9'
Reporte original en Volumenes: 'REPORTE #10', Reporte extraído: '10'
Reporte original en Volumenes: 'REPORTE #11', Reporte extraído: '11'
Reporte original en Volumenes: 'REPORTE #12', Reporte extraído: '12'
Reporte original en Volumenes: 'REPORTE #13', Reporte extraído: '13'
Reporte original en Volumenes: 'REPORTE #14', Reporte extraído: '14'
Reporte original en Volumenes: 'REPORTE #15', Report